<a href="https://colab.research.google.com/github/eTcilopp/ai_workshop_project/blob/master/ml/trading_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! pip install python-dotenv sqlalchemy psycopg2 -q

In [2]:
! pip install --upgrade pandas_ta -q

In [3]:
! pip install numpy==1.23.5 -q

In [4]:
from google.colab import userdata
from sqlalchemy import create_engine, MetaData, Table, select
from sqlalchemy.orm import declarative_base, sessionmaker
import numpy as np
import pandas as pd
import os

In [5]:
USER = 'postgres.kdakobaosmskerwwdmgx'
PASSWORD = userdata.get('ML_PROJECT_SUPABASE')  # Можно захардкодить
HOST ='aws-0-eu-north-1.pooler.supabase.com'
PORT = 5432
DBNAME ='postgres'

In [6]:
DATABASE_URL = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}?sslmode=require"

engine = create_engine(DATABASE_URL)


try:
    with engine.connect() as connection:
        print("Connection successful!")
except Exception as e:
    print(f"Failed to connect: {e}")

Connection successful!


In [7]:
Session = sessionmaker(bind=engine)
session = Session()

In [8]:
metadata = MetaData()
metadata.reflect(bind=engine)
ticker_table = Table('kandles_btc', metadata, autoload_with=engine)

In [20]:
stmp = select(ticker_table)
with engine.connect() as conn:
    df = pd.read_sql(stmp, conn)

In [10]:
session.close()

In [11]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,taker_buy_quote_asset_volume,ignore
0,2019-07-26 04:00:00+00:00,9694.63,9735.61,9686.02,9733.70,731.580978,2019-07-26 04:59:59.999000+00:00,7.104523e+06,9708,405.100424,3.934054e+06,0
1,2019-07-26 05:00:00+00:00,9733.72,9734.16,9701.69,9720.95,456.000146,2019-07-26 05:59:59.999000+00:00,4.431072e+06,7137,277.475348,2.695942e+06,0
2,2019-07-26 06:00:00+00:00,9720.41,9727.76,9657.00,9700.00,852.444378,2019-07-26 06:59:59.999000+00:00,8.264365e+06,11343,471.319259,4.570666e+06,0
3,2019-07-26 07:00:00+00:00,9698.73,9789.82,9683.91,9764.36,1459.969411,2019-07-26 07:59:59.999000+00:00,1.423650e+07,16927,722.051315,7.038039e+06,0
4,2019-07-26 08:00:00+00:00,9764.18,9803.88,9735.89,9791.51,961.067231,2019-07-26 08:59:59.999000+00:00,9.391734e+06,11861,510.650631,4.991497e+06,0


# Добавление признаков

In [12]:
from datetime import datetime, timedelta
import pandas_ta as ta

In [15]:
def get_extra_features(df):
  df['SMA 10'] = df['close'].rolling(window=10).mean()
  df['SMA 50'] = df['close'].rolling(window=50).mean()
  df['SMA 200'] = df['close'].rolling(window=200).mean()

  df['EMA10'] = df['close'].ewm(span=10, adjust=False).mean()

  df[['ADX_14', 'DMP_14', 'DMN_14']] = ta.adx(df['high'], df['low'], df['close'])

  df.ta.macd(close='close', fast=12, slow=26, signal=9, append=True)

  df['Price Change'] = df['close'].diff()
  df['Volume Change'] = df['volume'].diff()
  df['Daily Range'] = df['high'] - df['low']

  df['Close Lag 1'] = df['close'].shift(1)
  df['Close Lag 2'] = df['close'].shift(2)
  df['Close Lag 3'] = df['close'].shift(3)

  df['Volume Lag 1'] = df['volume'].shift(1)
  df['Volume Lag 2'] = df['volume'].shift(2)
  df['Volume Lag 3'] = df['volume'].shift(3)

  df['ATR'] = ta.atr(high=df['high'], low=df['low'], close=df['close'], length=14)

  df = df.reset_index()
  df['Day of the week'] = df['open_time'].dt.day_of_week
  df['Month'] = df['open_time'].dt.month

  df = df.drop(columns=['open_time'])

  df = df.dropna()
  return df

In [19]:
def add_target_column(df, percentage_grow=0.02):
  df['Open Next Day'] = df['open'].shift(-1)
  df['Close Next Day'] = df['close'].shift(-1)
  df['Target'] = np.where(
      (
          (df['Close Next Day'] > df['Open Next Day']) &
          ((df['Close Next Day'] - df['Open Next Day'])/df['Open Next Day'] > percentage_grow)
      ),
      1,
      0
  )
  df = df.drop(columns=['Open Next Day', 'Close Next Day'])
  df = df.dropna()
  return df

In [21]:
df_with_features = get_extra_features(df)
df_with_features_and_target = add_target_column(df_with_features)

In [22]:
df_with_features_and_target.head()

,index,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,...,Close Lag 1,Close Lag 2,Close Lag 3,Volume Lag 1,Volume Lag 2,Volume Lag 3,ATR,Day of the week,Month,Target
199,199,10768.02,10783.34,10731.02,10743.92,590.581024,2019-08-03 11:59:59.999000+00:00,6.354600e+06,8593,298.342381,...,10768.02,10763.38,10784.85,604.304767,1901.402933,1021.902050,97.927049,5,8,0
200,200,10741.97,10745.09,10662.15,10731.85,1574.336368,2019-08-03 12:59:59.999000+00:00,1.685823e+07,15576,714.851265,...,10743.92,10768.02,10763.38,590.581024,604.304767,1901.402933,96.856545,5,8,0
201,201,10733.47,10828.55,10732.73,10796.00,1872.452176,2019-08-03 13:59:59.999000+00:00,2.020798e+07,16621,953.945601,...,10731.85,10743.92,10768.02,1574.336368,590.581024,604.304767,96.845364,5,8,0
202,202,10795.20,10818.99,10775.00,10791.48,822.575092,2019-08-03 14:59:59.999000+00:00,8.879842e+06,9734,483.214811,...,10796.00,10731.85,10743.92,1872.452176,1574.336368,590.581024,93.069979,5,8,0
203,203,10791.94,10809.00,10675.50,10733.05,1473.725002,2019-08-03 15:59:59.999000+00:00,1.585002e+07,15057,670.121817,...,10791.48,10796.00,10731.85,822.575092,1872.452176,1574.336368,95.957839,5,8,0


In [23]:
df_with_features_and_target['Target'].value_counts()

,count
Target,
0,50261
1,490
